# ASHRAE GEPIII – Modeling

Per-meter LightGBM on `log1p(meter_reading)` with a time-based holdout.


In [1]:
import sys, pathlib, gc
sys.path.append(str(pathlib.Path.cwd().parent))

import numpy as np
import pandas as pd
import lightgbm as lgb

from src.features import build, FEATURES, CAT_FEATURES, rmsle

In [2]:
train = build("train")
train["target"] = np.log1p(train["meter_reading"].astype("float32"))

## Time-based split — last 30 days as validation


In [3]:
cutoff = train["timestamp"].max() - pd.Timedelta(days=30)
tr_idx = train["timestamp"] <= cutoff
val_idx = ~tr_idx

## Train one LightGBM per meter type


In [4]:
params = {
    "objective": "regression",
    "metric": "rmse",
    "learning_rate": 0.05,
    "num_leaves": 255,
    "feature_fraction": 0.85,
    "bagging_fraction": 0.85,
    "bagging_freq": 5,
    "min_data_in_leaf": 200,
    "verbose": -1,
}

models = {}
val_scores = {}

for meter in sorted(train["meter"].unique()):
    mask = train["meter"] == meter
    tr = train[mask & tr_idx]
    va = train[mask & val_idx]

    dtrain = lgb.Dataset(tr[FEATURES], tr["target"], categorical_feature=CAT_FEATURES)
    dval = lgb.Dataset(va[FEATURES], va["target"], categorical_feature=CAT_FEATURES, reference=dtrain)

    model = lgb.train(
        params,
        dtrain,
        num_boost_round=2000,
        valid_sets=[dtrain, dval],
        valid_names=["train", "val"],
        callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)],
    )
    pred = np.expm1(model.predict(va[FEATURES], num_iteration=model.best_iteration))
    val_scores[meter] = rmsle(va["meter_reading"].values, pred)
    models[meter] = model
    print(f"meter={meter} RMSLE={val_scores[meter]:.4f}")

print("overall val RMSLE:", np.mean(list(val_scores.values())))

Training until validation scores don't improve for 50 rounds
[100]	train's rmse: 0.418758	val's rmse: 0.49247
[200]	train's rmse: 0.351156	val's rmse: 0.485749
[300]	train's rmse: 0.33082	val's rmse: 0.482648
Early stopping, best iteration is:
[272]	train's rmse: 0.336191	val's rmse: 0.481678
meter=0 RMSLE=0.4816
Training until validation scores don't improve for 50 rounds
[100]	train's rmse: 0.886397	val's rmse: 1.11746
[200]	train's rmse: 0.800037	val's rmse: 1.09331
[300]	train's rmse: 0.763279	val's rmse: 1.08011
Early stopping, best iteration is:
[338]	train's rmse: 0.753038	val's rmse: 1.07173
meter=1 RMSLE=1.0674
Training until validation scores don't improve for 50 rounds
[100]	train's rmse: 1.0251	val's rmse: 0.978341
[200]	train's rmse: 0.955956	val's rmse: 0.973365
Early stopping, best iteration is:
[162]	train's rmse: 0.972778	val's rmse: 0.97017
meter=2 RMSLE=0.9700
Training until validation scores don't improve for 50 rounds
[100]	train's rmse: 1.09527	val's rmse: 1.4944


## Feature importance (electricity)


In [5]:
imp = pd.DataFrame({
    "feature": FEATURES,
    "gain": models[0].feature_importance(importance_type="gain"),
}).sort_values("gain", ascending=False)
imp.head(20)

,feature,gain
1,building_id,1.072048e+08
3,square_feet,9.406136e+07
0,site_id,4.847797e+06
18,month,4.079306e+06
16,hour,2.442748e+06
5,floor_count,1.905238e+06
2,primary_use,1.757082e+06
4,year_built,1.622016e+06
9,dew_temperature,1.512035e+06
17,weekday,1.313185e+06


## Predict on test and write submission


In [6]:
del train; gc.collect()

test = build("test")
preds = np.zeros(len(test), dtype="float32")
for meter, model in models.items():
    mask = (test["meter"] == meter).values
    if mask.any():
        preds[mask] = np.expm1(
            model.predict(test.loc[mask, FEATURES], num_iteration=model.best_iteration)
        )

preds = np.clip(preds, 0, None)
submission = pd.DataFrame({"row_id": test["row_id"], "meter_reading": np.round(preds, 4)})
submission.to_csv("../submission.csv", index=False)
submission.head()


,row_id,meter_reading
0,0,203.374405
1,1,89.096100
2,2,10.999700
3,3,225.779800
4,4,1398.262329
